In [6]:
"""
Converting_MatLabFile_To_Python.py

Python equivalent of Converting_MatLabFile_To_R.R

Reads a MATLAB .mat file containing EDA data (1328 timepoints x 23
participants), extracts the numeric matrix and participant IDs, assigns
the IDs as column names, and writes the result out to a CSV file.

Requires: scipy, pandas
    pip install scipy pandas
"""

import os
from pathlib import Path

import pandas as pd
from scipy.io import loadmat

# ---------------------------------------------------------------------------
# Set working directory to the folder this script lives in
# (equivalent of the rstudioapi current_path / setwd() calls in the R script)
# ---------------------------------------------------------------------------
#current_path = Path(__file__).resolve().parent
#os.chdir(current_path)

# ---------------------------------------------------------------------------
# Load the .mat file
# squeeze_me=True collapses unnecessary singleton dimensions
# struct_as_record=False loads MATLAB structs as objects w/ attribute access
# ---------------------------------------------------------------------------
mat_path = "../../Data/Raw/eda_5m32s_23ptcp_shift.mat"
data = loadmat(mat_path, squeeze_me=True, struct_as_record=False)

# Inspect available variable names (equivalent of str(data) in R)
print("Variables in .mat file:", [k for k in data.keys() if not k.startswith("__")])

# The R script accessed data$eda.5m32s.23ptcp.shift -- MATLAB variable names
# with dots get sanitized by R; in Python/scipy the original MATLAB variable
# name is preserved as the dict key.
var_name = "eda_5m32s_23ptcp_shift"
inner = data[var_name]

# ---------------------------------------------------------------------------
# Extract the numeric matrix (1328 timepoints x 23 participants)
# inner is expected to be a MATLAB struct/cell with the matrix as the first
# field/element (mirrors inner[[1]] in R)
# ---------------------------------------------------------------------------
if hasattr(inner, "_fieldnames"):
    # MATLAB struct -> first field
    first_field = inner._fieldnames[0]
    eda_matrix = getattr(inner, first_field)
else:
    # MATLAB cell array -> first element
    eda_matrix = inner[0]

# ---------------------------------------------------------------------------
# Extract participant IDs as a clean list of strings
# Mirrors: sapply(inner[[2]], function(x) x[[1]][1, 1])
# ---------------------------------------------------------------------------
if hasattr(inner, "_fieldnames"):
    second_field = inner._fieldnames[1]
    ids_raw = getattr(inner, second_field)
else:
    ids_raw = inner[1]

ids = []
for item in ids_raw:
    # Each item may itself be nested (e.g. a 1x1 cell/struct); drill down
    # until we get a plain string.
    val = item
    while hasattr(val, "__len__") and not isinstance(val, str):
        val = val[0]
    ids.append(str(val))

# Result: ['SUD_003_01_EDA', 'SUD_003_02_EDA', ...]

# ---------------------------------------------------------------------------
# Build the DataFrame with participant IDs as column names
# ---------------------------------------------------------------------------
eda_df = pd.DataFrame(eda_matrix, columns=ids)

print(eda_df.shape)          # should be (1328, 23)
print(eda_df.iloc[:, :5].head())
print(list(eda_df.columns))

# ---------------------------------------------------------------------------
# Write to CSV
# ---------------------------------------------------------------------------
df = eda_df
df.to_csv("MatLab_EDA_For_Python.csv", index=False)

Variables in .mat file: ['eda_5m32s_23ptcp_shift']
(1328, 23)
   SUD_003_01_EDA  SUD_003_02_EDA  SUD_003_04_EDA  SUD_003_06_EDA  \
0        0.356146        0.003844        0.017939        0.002563   
1        0.361270        0.005125        0.016657        0.001281   
2        0.354865        0.005125        0.017939        0.001281   
3        0.354865        0.006407        0.015376        0.001281   
4        0.353584        0.005125        0.016657        0.001281   

   SUD_003_08_EDA  
0        0.828076  
1        0.825513  
2        0.824232  
3        0.822950  
4        0.820388  
['SUD_003_01_EDA', 'SUD_003_02_EDA', 'SUD_003_04_EDA', 'SUD_003_06_EDA', 'SUD_003_08_EDA', 'SUD_003_09_EDA', 'SUD_003_10_EDA', 'SUD_003_11_EDA', 'SUD_003_12_EDA', 'SUD_003_14_EDA', 'SUD_003_15_EDA', 'SUD_003_16_EDA', 'SUD_003_17_EDA', 'SUD_003_18_EDA', 'SUD_003_20_EDA', 'SUD_003_21_EDA', 'SUD_003_22_EDA', 'SUD_003_23_EDA', 'SUD_003_24_EDA', 'SUD_003_27_EDA', 'SUD_003_28_EDA', 'SUD_003_29_EDA', 'SUD_0